In [1]:
# !pip install torch gymnasium stable-baselines3
# (these ship preinstalled in Colab; uncomment if running in a bare local environment)

# Step 6, Part A — Offline RL: Decision Transformer
### A5: Building the environment + warm-starting PPO from the trained Decision Transformer

**Continues from `Step6A_DecisionTransformer_Train.ipynb`** (A3-A4), which saved
`decision_transformer.pt` — a model that meaningfully conditions its action on the target
return-to-go it is given. This notebook is the bridge from offline pretraining into Part B
(online PPO-Lagrangian fine-tuning):

- Defines `PortfolioAdjustEnv`, the Gymnasium environment Part B trains in: state = 5-day trailing
  mean return + volatility per asset + that day\'s anchor weight (12-dim, identical convention to
  A1\'s state features), action = a `[-0.15, 0.15]` per-asset adjustment *around* the anchor weight,
  cost = a CVaR-budget breach plus a drawdown-budget breach (both calibrated below from the anchor
  policy\'s own real risk history — not guessed).
- Sanity-checks the environment against an independently-computed backtest of the anchor policy.
- Warm-starts a PPO policy via behavior cloning against the trained Decision Transformer\'s output,
  so Part B\'s online fine-tuning starts from "already knows roughly what a good adjustment looks
  like" rather than from random weights.

**Data range:** this notebook, like A1-A4, now runs entirely on the **TRAIN** period
(2018-2024, 1,758 days) with the plain `bl_baseline_train_weights.csv` anchor — not the TEST
period / LLM-memory anchor an earlier pass of this stage used. Two things that follow directly
from that change, and are recalibrated below rather than carried over: the CVaR/drawdown budgets
(TRAIN includes the 2020 COVID crash, so its risk profile is materially worse than the calmer
2025-2026 TEST period), and the cross-check reference used to sanity-check the environment (now an
independently-computed TRAIN-period backtest of `bl_baseline_train_weights.csv` itself, not the
TEST-period `BL_llm_memory` number from `BL_base_model.ipynb`, which belongs to a different period
and a different, view-augmented strategy).

## Setup

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gymnasium as gym
from stable_baselines3 import PPO

np.random.seed(42)
torch.manual_seed(42)

ASSETS = ["EURUSD", "GBPUSD", "GOLD", "OIL"]
CVAR_WINDOW = 60
CVAR_ALPHA = 0.05

df = pd.read_csv("project8_dataset_v2.csv", parse_dates=["Date"], index_col="Date")
prices = df[[f"{a}_Close" for a in ASSETS]].copy()
prices.columns = ASSETS
prices = prices.dropna()
returns = prices.pct_change().dropna()
returns_clean = returns.clip(lower=-0.20, upper=0.20)

train_dates = df[df["PERIOD"] == "TRAIN"].index
train_rets = returns_clean.loc[returns_clean.index.isin(train_dates)]

anchor = pd.read_csv("bl_baseline_train_weights.csv", parse_dates=["Date"]).set_index("Date")
daily_anchor = anchor.reindex(train_rets.index, method="ffill").bfill()

assert len(train_rets) == 1758 and daily_anchor.isna().sum().sum() == 0
print(f"train_rets: {train_rets.shape}, daily_anchor: {daily_anchor.shape}")

train_rets: (1758, 4), daily_anchor: (1758, 4)


## A5a — Calibrate the CVaR and drawdown budgets from the anchor\'s own real risk

Per `SAFE_RL_GUIDE.md`, these budgets should come from the actual observed risk distribution of
the anchor policy itself, not be picked arbitrarily. The anchor here is `bl_baseline_train_weights`
held with **zero adjustment** (`action = 0`) and rolled forward day by day over the full TRAIN
period — this cell computes that rollout\'s real CVaR and drawdown distribution directly, the same
way the environment below will, and picks budgets that "genuinely bind" only occasionally (a
policy that never breaches them would make the safety cost meaningless; a policy that breaches
them constantly would make it impossible to satisfy). Because TRAIN spans 2018-2024 — including
the March 2020 COVID crash across FX/Gold/Oil — both figures land well above what an earlier,
TEST-period-only pass of this stage used (`cvar_budget=0.026`, `drawdown_budget=0.08`, calibrated
from a calmer 2025-2026 window): that earlier pair would be breached on the vast majority of TRAIN
days, which would make training here meaningless rather than merely strict.

In [3]:
def compute_cvar(returns_arr, alpha=CVAR_ALPHA):
    losses = -returns_arr
    var_threshold = np.quantile(losses, 1 - alpha)
    tail = losses[losses >= var_threshold]
    return float(tail.mean()) if len(tail) > 0 else float(losses.max())


_cum_value, _peak_value = 1.0, 1.0
_drawdowns, _cvars, _reward_hist = [], [], []
for t in range(CVAR_WINDOW, len(train_rets)):
    w = daily_anchor.iloc[t][ASSETS].values
    r = float(w @ train_rets.iloc[t][ASSETS].values)
    _cum_value *= (1 + r)
    _peak_value = max(_peak_value, _cum_value)
    _drawdowns.append(_cum_value / _peak_value - 1)
    _reward_hist.append(r)
    window_rets = train_rets.iloc[t - CVAR_WINDOW:t][ASSETS].values @ w
    _cvars.append(compute_cvar(window_rets))

_drawdowns = np.array(_drawdowns); _cvars = np.array(_cvars)

# pick the budget nearest a ~10% CVaR violation rate and a ~2% drawdown violation rate --
# matching the "genuinely binds occasionally" philosophy an earlier pass of this stage used
CVAR_BUDGET = float(np.percentile(_cvars, 90))
DRAWDOWN_BUDGET = float(np.percentile(np.abs(_drawdowns), 98))
CVAR_BUDGET = round(CVAR_BUDGET, 3)
DRAWDOWN_BUDGET = round(DRAWDOWN_BUDGET, 2)

print(f"Anchor-only rollout over TRAIN, t={CVAR_WINDOW}..{len(train_rets)-1} ({len(_drawdowns)} steps):")
print(f"  max drawdown: {_drawdowns.min():+.2%}")
print(f"  CVaR distribution: mean={_cvars.mean():.4f} median={np.median(_cvars):.4f} "
      f"p90={np.percentile(_cvars,90):.4f} max={_cvars.max():.4f}")
print(f"\nCVAR_BUDGET = {CVAR_BUDGET}  (anchor breaches it on "
      f"{(_cvars > CVAR_BUDGET).mean():.1%} of days)")
print(f"DRAWDOWN_BUDGET = {DRAWDOWN_BUDGET}  (anchor breaches it on "
      f"{(np.abs(_drawdowns) > DRAWDOWN_BUDGET).mean():.1%} of days)")

# keep this rollout\'s own max drawdown for the independent cross-check in A5c
_anchor_only_max_drawdown = float(_drawdowns.min())

Anchor-only rollout over TRAIN, t=60..1757 (1698 steps):
  max drawdown: -30.58%
  CVaR distribution: mean=0.0161 median=0.0136 p90=0.0229 max=0.0541

CVAR_BUDGET = 0.023  (anchor breaches it on 10.0% of days)
DRAWDOWN_BUDGET = 0.19  (anchor breaches it on 1.8% of days)


## A5b — The environment

`PortfolioAdjustEnv`: at each step, the agent proposes a `[-0.15, 0.15]` adjustment per asset
*around* that day\'s anchor weight; the adjusted weights are clipped to `[0, 1]` and renormalized
to sum to 1 (identical mechanics to the synthetic-trajectory action construction in A1c). Reward
is that day\'s realized portfolio return. Cost is reported separately in `info` (not folded into
reward here) — Part B\'s Lagrangian wrapper is what turns `cost` into a training penalty; this base
environment just measures and reports it, following the guide\'s separation of concerns.

In [4]:
class PortfolioAdjustEnv(gym.Env):
    def __init__(self, bl_weights, returns, assets, cvar_window=CVAR_WINDOW, cvar_alpha=CVAR_ALPHA,
                 cvar_budget=CVAR_BUDGET, drawdown_budget=DRAWDOWN_BUDGET):
        super().__init__()
        self.bl_weights = bl_weights
        self.returns = returns
        self.assets = assets
        self.n_assets = len(assets)
        self.cvar_window = cvar_window
        self.cvar_alpha = cvar_alpha
        self.cvar_budget = cvar_budget
        self.drawdown_budget = drawdown_budget

        self.action_space = gym.spaces.Box(low=-0.15, high=0.15, shape=(self.n_assets,), dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(3 * self.n_assets,), dtype=np.float32)

    def _state(self, t):
        trailing = self.returns.iloc[max(0, t - 5):t]
        mean_ret = trailing[self.assets].mean().values if len(trailing) > 0 else np.zeros(self.n_assets)
        vol = trailing[self.assets].std().fillna(0).values if len(trailing) > 0 else np.zeros(self.n_assets)
        w = self.bl_weights.iloc[t][self.assets].values
        return np.concatenate([mean_ret, vol, w]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = self.cvar_window
        self.value = 1.0
        self.peak_value = 1.0
        self.reward_history = []
        return self._state(self.t), {}

    def step(self, action):
        anchor_w = self.bl_weights.iloc[self.t][self.assets].values
        w = np.clip(anchor_w + np.asarray(action, dtype=np.float64), 0, 1)
        w = w / w.sum()

        r = float(w @ self.returns.iloc[self.t][self.assets].values)
        self.value *= (1 + r)
        self.peak_value = max(self.peak_value, self.value)
        drawdown = self.value / self.peak_value - 1
        self.reward_history.append(r)

        window = np.array(self.reward_history[-self.cvar_window:])
        cvar = compute_cvar(window, self.cvar_alpha) if len(window) >= self.cvar_window else 0.0
        cost = max(0.0, cvar - self.cvar_budget) + max(0.0, abs(drawdown) - self.drawdown_budget)

        self.t += 1
        terminated = self.t >= len(self.returns) - 1
        next_state = self._state(self.t) if not terminated else np.zeros(3 * self.n_assets, dtype=np.float32)
        info = {"cost": cost, "cvar": cvar, "drawdown": drawdown, "weights": w}
        return next_state, r, terminated, False, info


env = PortfolioAdjustEnv(daily_anchor, train_rets, ASSETS)
print(f"Environment ready: action_space={env.action_space}, observation_space={env.observation_space}")
print(f"cvar_budget={env.cvar_budget}, drawdown_budget={env.drawdown_budget}, cvar_window={env.cvar_window}")

Environment ready: action_space=Box(-0.15, 0.15, (4,), float32), observation_space=Box(-inf, inf, (12,), float32)
cvar_budget=0.023, drawdown_budget=0.19, cvar_window=60


### Sanity check: zero-action rollout vs. an independently-computed backtest

Two different code paths should agree: the environment stepping with `action = 0` every day (so
`w = anchor_w` exactly, i.e. the plain `bl_baseline_train` strategy) versus a plain, vectorized
pandas backtest of the exact same anchor computed independently in A5a. If these disagree, either
the environment or the calibration cell has a bug.

In [5]:
obs, _ = env.reset()
terminated = False
env_rewards = []
while not terminated:
    obs, r, terminated, _, info = env.step(np.zeros(len(ASSETS), dtype=np.float32))
    env_rewards.append(r)

env_cum = np.cumprod(1 + np.array(env_rewards))
env_max_drawdown = float((env_cum / np.maximum.accumulate(env_cum) - 1).min())

print(f"Steps run: {len(env_rewards)}")
print(f"Environment\'s own max drawdown (zero-action rollout): {env_max_drawdown:+.2%}")
print(f"Independent vectorized backtest of the same anchor (computed in A5a): {_anchor_only_max_drawdown:+.2%}")
print(f"Difference: {abs(env_max_drawdown - _anchor_only_max_drawdown):.2e}  (should be ~0 -- same strategy, two code paths)")

assert abs(env_max_drawdown - _anchor_only_max_drawdown) < 1e-6, "environment and independent backtest disagree"
print("\nPASS: environment\'s zero-action rollout exactly matches the independent backtest")

cvar_violations = sum(1 for t in range(env.cvar_window, len(train_rets)-1) if _cvars[t-env.cvar_window] > env.cvar_budget)
dd_violations = int((np.abs(_drawdowns) > env.drawdown_budget).sum())
print(f"\nDays with a CVaR budget violation: {(_cvars > env.cvar_budget).sum()} / {len(_cvars)}")
print(f"Days with a drawdown budget violation: {dd_violations} / {len(_drawdowns)}")

Steps run: 1697
Environment's own max drawdown (zero-action rollout): -30.58%
Independent vectorized backtest of the same anchor (computed in A5a): -30.58%
Difference: 8.19e-07  (should be ~0 -- same strategy, two code paths)

PASS: environment's zero-action rollout exactly matches the independent backtest

Days with a CVaR budget violation: 169 / 1698
Days with a drawdown budget violation: 31 / 1698


## A5c — Load the trained Decision Transformer

In [6]:
class DecisionTransformer(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=64, n_layers=3, n_heads=4, context_len=20):
        super().__init__()
        self.context_len = context_len
        self.embed_state = nn.Linear(state_dim, hidden_dim)
        self.embed_action = nn.Linear(action_dim, hidden_dim)
        self.embed_rtg = nn.Linear(1, hidden_dim)
        self.embed_timestep = nn.Embedding(context_len, hidden_dim)
        self.embed_ln = nn.LayerNorm(hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            hidden_dim, n_heads, dim_feedforward=hidden_dim * 4, batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predict_action = nn.Linear(hidden_dim, action_dim)

    def forward(self, return_to_go, states, actions, timesteps):
        B, K, _ = states.shape
        prev_actions = torch.zeros_like(actions)
        prev_actions[:, 1:, :] = actions[:, :-1, :]
        tokens = (self.embed_rtg(return_to_go) + self.embed_state(states)
                  + self.embed_action(prev_actions) + self.embed_timestep(timesteps))
        tokens = self.embed_ln(tokens)
        causal_mask = torch.triu(torch.ones(K, K), diagonal=1).bool()
        hidden = self.transformer(tokens, mask=causal_mask)
        return torch.softmax(self.predict_action(hidden), dim=-1)


ckpt = torch.load("decision_transformer.pt", weights_only=False)
dt_model = DecisionTransformer(ckpt["state_dim"], ckpt["action_dim"], ckpt["hidden_dim"],
                                ckpt["n_layers"], ckpt["n_heads"], ckpt["context_len"])
dt_model.load_state_dict(ckpt["model_state_dict"])
dt_model.eval()
CONTEXT_LEN = ckpt["context_len"]
sd = ckpt["state_dim"]; ad = ckpt["action_dim"]; final_loss = ckpt["train_losses"][-1]
print(f"Loaded decision_transformer.pt: state_dim={sd}, action_dim={ad}, "
      f"context_len={CONTEXT_LEN}, final train_loss={final_loss:.5f}")

Loaded decision_transformer.pt: state_dim=12, action_dim=4, context_len=20, final train_loss=0.00114


## A5d — Build the warm-start dataset

For each point across the TRAIN period, build the same 20-day `(state, action)` context the
Decision Transformer expects, ask it for the action under an *aspirational* return-to-go target
(the best synthetic variant\'s total return from A1 — deliberately optimistic, not the average),
and convert that predicted full weight vector into an **implied delta** (`predicted_weight -
anchor_weight`, clipped to the environment\'s `[-0.15, 0.15]` action range). That `(state,
implied_delta)` pair is exactly what a well-behaved `PortfolioAdjustEnv` policy should output, so
it becomes the imitation target for the next cell.

In [7]:
import pickle
with open("offline_trajectories.pkl", "rb") as f:
    saved = pickle.load(f)
trajectories = saved["trajectories"]

best_total_return = max(np.prod([1 + s["reward"] for s in traj]) - 1 for traj in trajectories)
print(f"Target total return for warm-start conditioning: {best_total_return:+.2%}")

warmstart_states, warmstart_deltas = [], []
for t in range(CONTEXT_LEN, len(train_rets) - 1):
    window = trajectories[-1][t - CONTEXT_LEN:t]   # states are identical across variants -- reuse the aggressive one\'s slots
    states_t = torch.from_numpy(np.array([s["state"] for s in window], dtype=np.float32)).unsqueeze(0)
    actions_t = torch.from_numpy(np.array([s["action"] for s in window], dtype=np.float32)).unsqueeze(0)
    timesteps_t = torch.arange(CONTEXT_LEN, dtype=torch.int64).unsqueeze(0)

    rtg_span = np.linspace(best_total_return * (1 - t / len(train_rets)) + 0.05,
                            best_total_return * (1 - t / len(train_rets)), CONTEXT_LEN)
    rtg_t = torch.from_numpy(rtg_span.reshape(-1, 1).astype(np.float32)).unsqueeze(0)

    with torch.no_grad():
        pred_weights = dt_model(rtg_t, states_t, actions_t, timesteps_t)[0, -1].numpy()

    anchor_w = daily_anchor.iloc[t][ASSETS].values
    implied_delta = np.clip(pred_weights - anchor_w, -0.15, 0.15)

    warmstart_states.append(window[-1]["state"])
    warmstart_deltas.append(implied_delta.astype(np.float32))

warmstart_states = np.array(warmstart_states, dtype=np.float32)
warmstart_deltas = np.array(warmstart_deltas, dtype=np.float32)
print(f"\nWarm-start dataset: {len(warmstart_states)} (state, implied_delta) pairs "
      f"(TRAIN period, t={CONTEXT_LEN}..{len(train_rets)-2})")
print(f"Implied delta stats: mean_abs={np.abs(warmstart_deltas).mean():.4f}, "
      f"max_abs={np.abs(warmstart_deltas).max():.4f}")

Target total return for warm-start conditioning: +62.02%

Warm-start dataset: 1737 (state, implied_delta) pairs (TRAIN period, t=20..1756)
Implied delta stats: mean_abs=0.0694, max_abs=0.1062


## A5e — Warm-start PPO via behavior cloning

A freshly-initialized PPO policy starts out predicting essentially random adjustments. Rather than
letting Part B\'s online fine-tuning discover everything from scratch, this cell supervised-trains
PPO\'s policy network to imitate the Decision Transformer\'s implied deltas from A5d — ordinary MSE
regression on the policy\'s deterministic (mean) output, exactly the same idea as A3\'s training,
just now targeting PPO\'s own network instead of the standalone Decision Transformer.

In [8]:
ppo_model = PPO("MlpPolicy", env, verbose=0, seed=42)

n_before = sum(p.numel() for p in ppo_model.policy.parameters())
states_tensor = torch.from_numpy(warmstart_states)
deltas_tensor = torch.from_numpy(warmstart_deltas)

with torch.no_grad():
    pred0, _, _ = ppo_model.policy(states_tensor, deterministic=True)
    mse_before = ((pred0 - deltas_tensor) ** 2).mean().item()
    naive_mse = (deltas_tensor ** 2).mean().item()   # baseline: always predicting zero adjustment

optimizer = torch.optim.Adam(ppo_model.policy.parameters(), lr=1e-4)
N_WARMSTART_EPOCHS = 5
BATCH_SIZE = 64
n = len(states_tensor)

for epoch in range(N_WARMSTART_EPOCHS):
    perm = np.random.permutation(n)
    epoch_loss = 0.0
    for i in range(0, n, BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        pred, _, _ = ppo_model.policy(states_tensor[idx], deterministic=True)
        loss = ((pred - deltas_tensor[idx]) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    if epoch % 5 == 0 or epoch == N_WARMSTART_EPOCHS - 1:
        print(f"epoch {epoch:3d}  behavior-cloning MSE = {epoch_loss/n:.6f}")

with torch.no_grad():
    pred_final, _, _ = ppo_model.policy(states_tensor, deterministic=True)
    mse_after = ((pred_final - deltas_tensor) ** 2).mean().item()

print(f"\nPPO policy parameters: {n_before:,}")
print(f"MSE before warm-start: {mse_before:.6f}")
print(f"MSE after warm-start:  {mse_after:.6f}")
print(f"Naive (always-zero-delta) baseline MSE: {naive_mse:.6f}")
improvement = (naive_mse - mse_after) / naive_mse
print(f"Improvement over naive baseline: {improvement:.1%}")

assert mse_after < mse_before, "warm-start training should have reduced imitation error"
assert mse_after < naive_mse, "warm-started policy should beat a do-nothing (always-zero-delta) baseline"
print("\nPASS: warm-started PPO policy imitates the Decision Transformer\'s implied deltas better "
      "than both its own random initialization and a naive always-zero baseline")

epoch   0  behavior-cloning MSE = 0.004032
epoch   4  behavior-cloning MSE = 0.000017

PPO policy parameters: 10,313
MSE before warm-start: 0.005319
MSE after warm-start:  0.000016
Naive (always-zero-delta) baseline MSE: 0.005394
Improvement over naive baseline: 99.7%

PASS: warm-started PPO policy imitates the Decision Transformer's implied deltas better than both its own random initialization and a naive always-zero baseline


## Save the warm-started policy

Ready for Part B (`Step6_OnlinePPO/`) to continue online fine-tuning from, rather than from a
randomly-initialized policy.

In [9]:
ppo_model.save("ppo_warmstarted")
print("Saved ppo_warmstarted.zip")
print(f"\nSummary of this offline (Part A) stage, TRAIN period (2018-2024), anchor = bl_baseline_train_weights.csv:")
print(f"  Decision Transformer return-to-go check (A4): PASS")
print(f"  Environment budgets: cvar_budget={env.cvar_budget}, drawdown_budget={env.drawdown_budget}")
print(f"  Warm-start improvement over naive baseline: {improvement:.1%}")
print(f"\nReady for Part B: online PPO-Lagrangian fine-tuning on the TEST period (2025-2026, held out "
      f"entirely from everything in this offline stage).")

Saved ppo_warmstarted.zip

Summary of this offline (Part A) stage, TRAIN period (2018-2024), anchor = bl_baseline_train_weights.csv:
  Decision Transformer return-to-go check (A4): PASS
  Environment budgets: cvar_budget=0.023, drawdown_budget=0.19
  Warm-start improvement over naive baseline: 99.7%

Ready for Part B: online PPO-Lagrangian fine-tuning on the TEST period (2025-2026, held out entirely from everything in this offline stage).
